# Gemini Gateway: end-to-end demo

This notebook exercises the complete deployed flow:

`Notebook -> private Cloud Run gateway -> Cloud Tasks -> Gemini -> Firestore -> notebook`

The gateway immediately returns a job ID. The notebook polls the status endpoint until the Firestore-backed job is completed or failed.

## Prerequisites

For a local Jupyter environment, authenticate the Google Cloud CLI:

```bash
gcloud auth login
gcloud config set project YOUR_PROJECT_ID
```

The active user must be allowed to impersonate `gemini-gateway-dispatcher`, which needs Cloud Run Invoker access. When running inside Google Cloud without `gcloud`, attach a service account with Cloud Run Invoker access.

In [ ]:
%pip install -q requests google-auth

In [ ]:
import os
import shutil
import subprocess
import time
import uuid

import requests
from IPython.display import JSON, Markdown, display

PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT", "YOUR_PROJECT_ID")
SERVICE_URL = os.getenv("GEMINI_GATEWAY_URL", "YOUR_CLOUD_RUN_URL").rstrip("/")
DISPATCHER_SERVICE_ACCOUNT = os.getenv(
    "GEMINI_GATEWAY_CALLER_SERVICE_ACCOUNT",
    f"gemini-gateway-dispatcher@{PROJECT_ID}.iam.gserviceaccount.com",
)
POLL_TIMEOUT_SECONDS = 300

if PROJECT_ID == "YOUR_PROJECT_ID" or SERVICE_URL == "YOUR_CLOUD_RUN_URL":
    raise RuntimeError(
        "Set GOOGLE_CLOUD_PROJECT and GEMINI_GATEWAY_URL in your environment, "
        "or replace the YOUR_* placeholders in this configuration cell."
    )

print(f"Gateway: {SERVICE_URL}")

## Authenticate to the private Cloud Run service

Locally, the helper uses your `gcloud` login to impersonate the narrowly scoped dispatcher service account and creates a short-lived token whose audience is exactly the Cloud Run URL. Inside Google Cloud, it can use the attached service account. The token is kept only in memory and is never displayed.

In [ ]:
def get_identity_token(audience: str) -> str:
    if shutil.which("gcloud"):
        completed = subprocess.run(
            [
                "gcloud",
                "auth",
                "print-identity-token",
                f"--impersonate-service-account={DISPATCHER_SERVICE_ACCOUNT}",
                f"--audiences={audience}",
                "--include-email",
            ],
            check=True,
            capture_output=True,
            text=True,
        )
        return completed.stdout.strip()

    from google.auth.transport.requests import Request as GoogleAuthRequest
    from google.oauth2 import id_token as google_id_token

    return google_id_token.fetch_id_token(GoogleAuthRequest(), audience)


identity_token = get_identity_token(SERVICE_URL)
HEADERS = {
    "Authorization": f"Bearer {identity_token}",
    "Content-Type": "application/json",
}

health = requests.get(f"{SERVICE_URL}/health", headers=HEADERS, timeout=30)
health.raise_for_status()
print("Authenticated successfully:", health.json())

## Submit a Gemini request

The idempotency key prevents accidental duplicate Gemini calls if the submit operation is retried.

In [ ]:
payload = {
    "prompt": (
        "Explain in three concise bullet points why a centralized request queue "
        "is useful when multiple services call Gemini."
    ),
    "generation_config": {
        "temperature": 0.2,
        "max_output_tokens": 800,
    },
    "metadata": {
        "caller": "gateway-demo-notebook",
    },
}

submit_headers = {
    **HEADERS,
    "Idempotency-Key": f"notebook-demo-{uuid.uuid4()}",
}
response = requests.post(
    f"{SERVICE_URL}/v1/requests",
    headers=submit_headers,
    json=payload,
    timeout=30,
)
response.raise_for_status()
submitted_job = response.json()
job_id = submitted_job["id"]

display(JSON(submitted_job))

## Poll Firestore-backed job status

The polling interval gradually increases to avoid unnecessary traffic. The final document contains either `result` or `error`.

In [ ]:
def wait_for_job(job_id: str, timeout_seconds: int = 300) -> dict:
    deadline = time.monotonic() + timeout_seconds
    delay = 1.0
    previous_status = None

    while time.monotonic() < deadline:
        response = requests.get(
            f"{SERVICE_URL}/v1/requests/{job_id}",
            headers=HEADERS,
            timeout=30,
        )
        response.raise_for_status()
        job = response.json()
        status = job["status"]

        if status != previous_status:
            print(f"Status: {status}")
            previous_status = status

        if status == "completed":
            return job
        if status in {"failed", "enqueue_failed"}:
            raise RuntimeError(f"Gateway job failed: {job.get('error')}")

        time.sleep(delay)
        delay = min(delay * 1.5, 5.0)

    raise TimeoutError(f"Job {job_id} did not finish within {timeout_seconds} seconds")


completed_job = wait_for_job(job_id, POLL_TIMEOUT_SECONDS)
display(JSON(completed_job))

## Display the Gemini result

In [ ]:
result = completed_job["result"]
display(Markdown(result.get("text") or "_Gemini returned no visible text._"))

print("Token usage:")
display(JSON(result.get("usage") or {}))

## Confirm that the result remains available

This reads the same completed result again from the gateway. The worker does not call Gemini again.

In [ ]:
stored_response = requests.get(
    f"{SERVICE_URL}/v1/requests/{job_id}",
    headers=HEADERS,
    timeout=30,
)
stored_response.raise_for_status()
stored_job = stored_response.json()

assert stored_job["status"] == "completed"
assert stored_job["result"] == completed_job["result"]
print("Stored result retrieved successfully from Firestore-backed job state.")

## Optional: demonstrate several queued requests

Change `RUN_BATCH_DEMO` to `True` to submit five additional paid Gemini requests. They will pass through the configured queue limits of one dispatch per second and two concurrent workers.

In [ ]:
RUN_BATCH_DEMO = False

if RUN_BATCH_DEMO:
    batch_jobs = []
    for number in range(1, 6):
        response = requests.post(
            f"{SERVICE_URL}/v1/requests",
            headers={
                **HEADERS,
                "Idempotency-Key": f"notebook-batch-{uuid.uuid4()}",
            },
            json={
                "prompt": f"Return only the square of {number}.",
                "generation_config": {"temperature": 0, "max_output_tokens": 100},
                "metadata": {"caller": "gateway-demo-batch", "number": number},
            },
            timeout=30,
        )
        response.raise_for_status()
        batch_jobs.append(response.json())

    print("Submitted jobs:", [job["id"] for job in batch_jobs])
    batch_results = [wait_for_job(job["id"], POLL_TIMEOUT_SECONDS) for job in batch_jobs]
    display(JSON(batch_results))
else:
    print("Batch demo skipped. Set RUN_BATCH_DEMO = True to run it.")